<a href="https://colab.research.google.com/github/2470024/data-science-portfolio/blob/main/CP5_%E9%AB%99%E7%94%B0%E6%82%A0%E6%96%97.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd
import io
import ipywidgets as widgets
from IPython.display import display, clear_output

# ファイルをアップロード
uploaded = files.upload()

# アップロードされたファイル名を自動取得して読み込む
df=None

for filename in uploaded.keys():
    df = pd.read_csv(io.BytesIO(uploaded[filename]), skiprows=[1])
    df.set_index(df.columns[0], inplace=True)  # 1列目をインデックスに

    print(f"読み込んだファイル: {filename}")
    display(df.head())  # 最初の数行を表示
    # display(df.loc[4]) # インデックスが4の行を表示

# 列名を表示
print("列名（カラム名）一覧:")
print(df.columns.tolist())

# --- ボタンを3つ作成（カラム名を仮に指定。実際のカラム名に合わせて変更してください） ---
# ※ここでは "Height", "Weight", "Score" という列があると仮定しています

# カラム名をユーザーのデータに合わせて変更してください
col1 = "10m走"
col2 = "20m走"
col3 = "CMJ"

# ボタンを作成
button1 = widgets.Button(description=f"{col1} の平均")
button2 = widgets.Button(description=f"{col2} の平均")
button3 = widgets.Button(description=f"{col3} の平均")

output = widgets.Output()

# ボタンのクリック時の処理
def show_average(col_name):
    with output:
        clear_output()
        avg = df[col_name].mean()
        print(f"{col_name} の平均値: {avg:.2f}")

button1.on_click(lambda b: show_average(col1))
button2.on_click(lambda b: show_average(col2))
button3.on_click(lambda b: show_average(col3))

# ボタンを表示
display(widgets.HBox([button1, button2, button3]))
display(output)


Saving PhysicalTest_25.csv to PhysicalTest_25.csv
読み込んだファイル: PhysicalTest_25.csv


,性別,10m走,20m走,CMJ,DJ-接地時間,DJ-跳躍高,DJ-RSI,HG右,HG左,HG合計
ID,,,,,,,,,,
1,男,1.85,3.08,57.7,0.235,51.0,2.17,3.47,3.31,6.78
2,女,2.02,3.55,30.3,0.218,30.5,1.40,4.23,4.06,8.29
3,男,1.97,3.42,31.5,0.228,24.5,1.07,4.70,5.17,9.87
4,男,1.83,3.08,40.4,0.195,17.2,0.85,4.04,3.98,8.02
5,女,2.05,3.63,25.2,0.295,19.9,0.67,3.74,4.01,7.75


列名（カラム名）一覧:
['性別', '10m走', '20m走', 'CMJ', 'DJ-接地時間', 'DJ-跳躍高', 'DJ-RSI', 'HG右', 'HG左', 'HG合計']


Output()

In [ ]:
# ============================================================
# 【目標ポジション対応版】バレーボール ポジション診断ツール
# ============================================================
import sys
import io
import pandas as pd
from google.colab import files
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ------------------------------------------------------------
# 1. 診断データ・アドバイスデータの定義
# ------------------------------------------------------------
POSITIONS = ["アウトサイドヒッター", "オポジット", "ミドルブロッカー", "セッター", "リベロ"]

POSITION_WEIGHTS = {
    "アウトサイドヒッター": {"DJ-RSI": 3, "CMJ": 3, "DJ-跳躍高": 2, "HG合計": 2, "10m走": 1},
    "オポジット": {"CMJ": 3, "DJ-跳躍高": 3, "DJ-RSI": 3, "DJ-接地時間": 1},
    "ミドルブロッカー": {"DJ-接地時間": 3, "DJ-RSI": 3, "HG合計": 2, "10m走": 2, "CMJ": 1},
    "セッター": {"HG合計": 3, "DJ-RSI": 2, "10m走": 2, "CMJ": 1},
    "リベロ": {"HG合計": 3, "10m走": 3, "20m走": 3, "DJ-接地時間": 1},
}

LOWER_IS_BETTER = {"10m走", "20m走", "DJ-接地時間", "HG右", "HG左", "HG合計"}

REFERENCE = {
    "10m走": 1.90, "20m走": 3.20, "CMJ": 48.0, "DJ-接地時間": 0.21,
    "DJ-跳躍高": 36.0, "DJ-RSI": 1.70, "HG右": 9.0, "HG左": 9.3, "HG合計": 18.3,
}

STD_DEV = {
    "10m走": 0.12, "20m走": 0.18, "CMJ": 6.0, "DJ-接地時間": 0.04,
    "DJ-跳躍高": 5.0, "DJ-RSI": 0.40, "HG右": 1.0, "HG左": 1.0, "HG合計": 1.8,
}

TRAINING_ADVICE = {
    "アウトサイドヒッター": [
        "【スパイクジャンプ強化】ハーフスクワット + ジャンプスクワット（3セット×8回）でDJ-RSIを高める",
        "【助走踏み切り練習】バウンディング（連続跳躍）でSSC（伸張-短縮サイクル）を強化する",
        "【全方向フットワーク】ラダートレーニング + 多方向サイドステップでレセプション後の対応力を上げる",
    ],
    "オポジット": [
        "【最大跳躍力強化】バーベルスクワット + クリーンで下肢の爆発力を最大化する",
        "【ボックスジャンプ】40〜60cmの台へのジャンプを繰り返しCMJを伸ばす",
        "【連続スパイク練習】インターバルジャンプ（10回×3セット）で高さを維持する持久力を養う",
    ],
    "ミドルブロッカー": [
        "【デプスジャンプ】30cmの台から飛び降り即座にジャンプ。DJ-接地時間の短縮が最優先",
        "【左右ブロック移動】コートの幅を使ったサイドステップ＋ジャンプを繰り返しHGを改善する",
        "【短距離ダッシュ】10mダッシュ×10本でブロード移動のスプリント力を高める",
    ],
    "セッター": [
        "【多方向敏捷性】ヘキサゴントレーニング（正六角形の内外を連続ジャンプ）でHG値を直接改善",
        "【反応トレーニング】ランダムな方向への素早いスプリントでボール落下点への到達スピードを上げる",
        "【ジャンプトス練習】実際のセッタージャンプを反復しDJ-RSIを高める",
    ],
    "リベロ": [
        "【HG直接改善】ヘキサゴントレーニング（左右両方向）を週3回実施しHG合計を短縮する",
        "【10m/20mダッシュ強化】スタートダッシュ練習（反応スタート付き）で守備の出足を鋭くする",
        "【低姿勢フットワーク】低いスプリットステップ + 素早いダイブで守備範囲を広げる",
    ],
}

TARGET_POSITION_MESSAGES = {
    "アウトサイドヒッター": "オフェンス・ディフェンスの両面で高いレベルが求められるポジションです。特にスピーディな助走から鋭く踏み切る能力（DJ-RSI）と、レセプション後にすぐ攻撃参加するための敏捷性（HG合計）が求められます。",
    "オポジット": "チームの主砲として、守備負担が少ない分、純粋な「高さとパワー」が武器になります。CMJとDJ-跳躍高を限界まで伸ばす下肢ストレングストレーニングを最優先にしてください。",
    "ミドルブロッカー": "相手のスパイクに素早く反応するネット際の「クイックネス」が命です。ジャンプの高さ以上に、着地から次のジャンプへの切り替え（DJ-接地時間）の短縮と、左右へのブロックステップの速さ（HG合計）を鍛えましょう。",
    "セッター": "あらゆる状況から正確なトスを供給するため、コート内を縦横無尽に素早く動くフットワークが必須です。全方向への切り返しスピード（HG合計）と、落下点へ滑り込む走力（10m走）を集中的に磨きます。",
    "リベロ": "守備の専門職であり、ジャンプ力よりも「移動スピードと反応の鋭さ」が全てです。10m・20mのダッシュ力、およびヘキサゴン（HG合計）の数値を圧倒的に高めることで、守備範囲が格段に広がります。"
}

# ------------------------------------------------------------
# 2. ロジック関数群
# ------------------------------------------------------------
def get_deviation_score(col, value):
    ref = REFERENCE.get(col)
    std = STD_DEV.get(col)
    if ref is None or std is None or pd.isna(value):
        return 50.0
    diff = (value - ref) / std
    if col in LOWER_IS_BETTER:
        diff = -diff
    return round(50 + diff * 10, 1)

def calc_position_scores(row):
    scores = {}
    for pos, weights in POSITION_WEIGHTS.items():
        total, weight_sum = 0.0, 0
        for col, w in weights.items():
            try: val = float(row.get(col, float("nan")))
            except: val = float("nan")
            total += get_deviation_score(col, val) * w
            weight_sum += w
        scores[pos] = round(total / weight_sum, 1) if weight_sum > 0 else 50.0
    return scores

def grade_badge(score):
    if score >= 65: return '<span style="background:#28a745;color:white;padding:2px 8px;border-radius:4px;font-size:0.85em;">◎ 優秀</span>'
    if score >= 55: return '<span style="background:#007bff;color:white;padding:2px 8px;border-radius:4px;font-size:0.85em;">○ 良好</span>'
    if score >= 45: return '<span style="background:#ffc107;color:black;padding:2px 8px;border-radius:4px;font-size:0.85em;">△ 平均</span>'
    return '<span style="background:#dc3545;color:white;padding:2px 8px;border-radius:4px;font-size:0.85em;">▽ 要強化</span>'

# メインレポート生成（基本カルテ + 目標ポジションへのギャップアドバイス）
def generate_html_report(row, target_pos=None):
    no = row.get("ID", "?")
    gender = row.get("性別", "-")

    scores = calc_position_scores(row)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    best_pos, best_score = ranked[0]

    # 特徴分析用の指標
    base_metrics = POSITION_WEIGHTS[best_pos]
    strengths = [k for k in base_metrics if get_deviation_score(k, row.get(k)) >= 55]
    weaknesses = [k for k in base_metrics if get_deviation_score(k, row.get(k)) < 45]

    html = f"""
    <style>
        .app-card {{ font-family: 'Helvetica Neue', Arial, sans-serif; background: #ffffff; padding: 25px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); margin: 10px 0; color: #333; border-left: 6px solid #1a73e8; }}
        .app-header {{ border-bottom: 2px solid #f1f3f4; padding-bottom: 15px; margin-bottom: 20px; }}
        .app-title {{ font-size: 22px; font-weight: bold; color: #1a73e8; }}
        .pos-table {{ width: 100%; border-collapse: collapse; margin: 15px 0; }}
        .pos-table th, .pos-table td {{ padding: 10px; text-align: left; border-bottom: 1px solid #f1f3f4; }}
        .pos-table th {{ background: #f8f9fa; color: #5f6368; font-weight: 600; }}
        .bar-container {{ background: #e8eaed; border-radius: 4px; height: 12px; width: 130px; display: inline-block; overflow: hidden; vertical-align: middle; }}
        .bar-fill {{ background: #34a853; height: 100%; }}
        .bar-fill.secondary {{ background: #4285f4; }}
        .section-title {{ font-size: 16px; font-weight: bold; margin-top: 25px; margin-bottom: 10px; color: #202124; border-bottom: 1px solid #e8eaed; padding-bottom: 5px; }}
        .advice-box {{ background: #f8f9fa; border-radius: 8px; padding: 12px; margin: 8px 0; border-left: 4px solid #34a853; font-size:13px; }}
        .target-box {{ background: #fff8e1; border-radius: 8px; padding: 15px; margin: 20px 0 0 0; border: 1px solid #ffe082; border-left: 6px solid #ffb300; }}
        .metric-grid {{ display: flex; flex-wrap: wrap; gap: 8px; margin: 10px 0; }}
        .metric-tag {{ padding: 6px 12px; border-radius: 20px; font-size: 13px; font-weight: 500; }}
        .gap-table {{ width: 100%; border-collapse: collapse; margin-top: 10px; background:#fff; border-radius:6px; overflow:hidden; }}
        .gap-table th, .gap-table td {{ padding: 8px 12px; font-size: 13px; text-align: left; border-bottom: 1px solid #f1f3f4; }}
        .gap-table th {{ background:#fff3e0; color:#e65100; }}
    </style>

    <div class="app-card">
        <div class="app-header">
            <div class="app-title">🏐 診断結果: 選手ID {no} （{gender}）</div>
        </div>

        <div style="display: flex; gap: 20px; flex-wrap: wrap;">
            <!-- 左側：ランキング -->
            <div style="flex: 1; min-width: 300px;">
                <div class="section-title">🏆 ポジション適性ランキング</div>
                <table class="pos-table">
                    <thead><tr><th>順位</th><th>ポジション</th><th>スコア</th><th>分布</th></tr></thead>
                    <tbody>
    """

    for i, (pos, sc) in enumerate(ranked, 1):
        pct = max(0, min(100, (sc - 30) * 2))
        bar_class = "bar-fill" if i == 1 else "bar-fill secondary"
        mark = " ⭐ 最適" if i == 1 else ""
        html += f"""
            <tr>
                <td>{i}位</td>
                <td><strong>{pos}</strong>{mark}</td>
                <td>{sc}点 ({grade_badge(sc)})</td>
                <td><div class="bar-container"><div class="{bar_class}" style="width: {pct}%;"></div></div></td>
            </tr>
        """

    html += f"""
                    </tbody>
                </table>
            </div>

            <!-- 右側：現在の強み・弱み -->
            <div style="flex: 1; min-width: 300px;">
                <div class="section-title">📊 身体能力の特徴 (最適ポジション基準)</div>
                <p style="margin-bottom:5px;"><strong>現在の強み項目:</strong></p>
                <div class="metric-grid">
                    {"".join([f'<div class="metric-tag" style="color:#28a745; background:#e8f5e9;">💪 {m}</div>' for m in strengths]) if strengths else '<div class="metric-tag" style="background:#f1f3f4;">バランス型（突出した指標はなし）</div>'}
                </div>

                <p style="margin-top:15px; margin-bottom:5px;"><strong>最適位への推奨トレーニング:</strong></p>
                {"".join([f'<div class="advice-box">{tip}</div>' for tip in TRAINING_ADVICE[best_pos]])}
            </div>
        </div>
    """

    # ── 🌟 新機能: 目標ポジションへのアドバイス・ギャップ分析 ──
    if target_pos:
        target_score = scores.get(target_pos, 50.0)
        is_best = (target_pos == best_pos)

        html += f"""
        <div class="target-box">
            <h3 style="margin-top:0; color:#b78103; display:flex; align-items:center; gap:5px;">
                🎯 目標ポジションシミュレーション: 【{target_pos}】
            </h3>
            <p style="margin: 5px 0;">現在の目標適性スコア: <strong>{target_score}点</strong> ({grade_badge(target_score)})
            { '<span style="color:#28a745; font-weight:bold; margin-left:10px;">(※現在、最も適性が高いポジションです！)</span>' if is_best else '<span style="color:#e65100; font-weight:bold; margin-left:10px;">(※適性向上への課題があります)</span>'}</p>
            <p style="font-size:13px; color:#555; background:#fff; padding:10px; border-radius:6px; border:1px solid #ffe082; line-height:1.5;">
                {TARGET_POSITION_MESSAGES[target_pos]}
            </p>
        """

        if not is_best:
            html += """
            <p style="margin-top:15px; margin-bottom:5px; font-weight:bold; color:#e65100;">⚠️ 目標ポジションに就くためのパラメーターギャップ分析:</p>
            <table class="gap-table">
                <thead>
                    <tr><th>必要指標</th><th>重要度</th><th>あなたの偏差値</th><th>評価</th><th>アドバイス</th></tr>
                </thead>
                <tbody>
            """
            # 目標ポジションの重要指標ごとに現在の数値を評価
            for metric, weight in sorted(POSITION_WEIGHTS[target_pos].items(), key=lambda x: x[1], reverse=True):
                m_score = get_deviation_score(metric, row.get(metric))
                weight_label = "🔥 最重要(x3)" if weight == 3 else ("⭐ 重要(x2)" if weight == 2 else "○ 必要(x1)")

                if m_score >= 55:
                    status = '<span style="color:green; font-weight:bold;">クリア</span>'
                    adv_text = "十分な能力です。維持しましょう。"
                elif m_score >= 45:
                    status = '<span style="color:#ff9800; font-weight:bold;">あと少し</span>'
                    adv_text = "平均レベルです。もう一歩強化が必要です。"
                else:
                    status = '<span style="color:red; font-weight:bold;">要強化</span>'
                    adv_text = "最優先の課題です。重点的なトレーニングが必要です。"

                html += f"""
                    <tr>
                        <td><strong>{metric}</strong></td>
                        <td>{weight_label}</td>
                        <td>{m_score}</td>
                        <td>{status}</td>
                        <td>{adv_text}</td>
                    </tr>
                """
            html += "</tbody></table>"

            html += f"""
            <p style="margin-top:15px; margin-bottom:5px; font-weight:bold; color:#b78103;">🏋️ 【{target_pos}】に近づくための特化トレーニングメニュー:</p>
            """
            for tip in TRAINING_ADVICE[target_pos]:
                html += f'<div class="advice-box" style="background:#fff; border-left:4px solid #ffb300;">{tip}</div>'

        html += "</div>"

    html += "</div>"
    return html

# ------------------------------------------------------------
# 3. アプリケーション本体（UI設計）
# ------------------------------------------------------------
class VolleyballApp:
    def __init__(self):
        self.df = None
        self.all_results = {}

        self.title_ui = widgets.HTML("""
            <div style="background: linear-gradient(135deg, #1a73e8, #0d47a1); color: white; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 20px;">
                <h1 style="margin: 0; font-size: 26px;">🏐 バレーボール ポジション診断ダッシュボード</h1>
                <p style="margin: 5px 0 0 0; opacity: 0.9;">適性診断 ＆ 目標ポジションへのギャップ分析・トレーニング指導機能</p>
            </div>
        """)

        # タブ1: データインポート
        self.upload_btn = widgets.Button(description="📂 PhysicalTest_25.csv をアップロード", button_style="primary", layout=widgets.Layout(width="280px"))
        self.upload_btn.on_click(self.upload_csv)
        self.import_status = widgets.HTML("<p style='color:#666; margin-top:10px;'>「PhysicalTest_25.csv」ファイルを選択してアップロードしてください。</p>")
        self.preview_output = widgets.Output()

        self.tab1_view = widgets.VBox([
            widgets.HTML("<h3>💡 ステップ 1: データの準備</h3>"),
            self.upload_btn,
            self.import_status,
            self.preview_output
        ])

        # タブ2: 個人診断 ＆ 目標シミュレーション
        self.player_select = widgets.Dropdown(description="① 選手を選択:", disabled=True, layout=widgets.Layout(width="280px"))
        self.player_select.observe(self.on_ui_change, names='value')

        self.target_select = widgets.Dropdown(description="② 目標職種:", options=POSITIONS, disabled=True, layout=widgets.Layout(width="280px"))
        self.target_select.observe(self.on_ui_change, names='value')

        self.report_output = widgets.Output()

        self.tab2_view = widgets.VBox([
            widgets.HTML("<h3>👤 選手別 詳細カルテ ＆ 目標ポジション分析</h3>"),
            widgets.HTML("<p style='color:#666; font-size:13px;'>選手を選択した後、「もし別のポジションを目指すなら？」というシミュレーションが下部のセレクトボックスから行えます。</p>"),
            widgets.HBox([self.player_select, self.target_select], layout=widgets.Layout(gap="20px", margin="10px 0")),
            widgets.Box([self.report_output], layout=widgets.Layout(margin="10px 0 0 0"))
        ])

        # タブ3: 全体比較
        self.download_results_btn = widgets.Button(description="📥 診断結果CSVをダウンロード", button_style="success", disabled=True, layout=widgets.Layout(width="260px"))
        self.download_results_btn.on_click(self.download_processed_csv)
        self.matrix_output = widgets.Output()

        self.tab3_view = widgets.VBox([
            widgets.HTML("<h3>📊 チーム全体の適性マトリクス</h3>"),
            self.download_results_btn,
            widgets.Box([self.matrix_output], layout=widgets.Layout(margin="15px 0 0 0"))
        ])

        self.tabs = widgets.Tab()
        self.tabs.children = [self.tab1_view, self.tab2_view, self.tab3_view]
        self.tabs.set_title(0, '📂 データ読込')
        self.tabs.set_title(1, '👤 個人診断 & 希望強化')
        self.tabs.set_title(2, '📊 チーム比較')

    def display_app(self):
        display(self.title_ui)
        display(self.tabs)

    def upload_csv(self, b):
        self.import_status.value = "<p style='color:#1a73e8; font-weight:bold;'>⌛ ファイル選択ウィンドウを待機中...</p>"
        with self.preview_output:
            clear_output()
            try:
                uploaded = files.upload()
                if not uploaded:
                    self.import_status.value = "<p style='color:red;'>⚠️ アップロードがキャンセルされました。</p>"
                    return

                filename = list(uploaded.keys())[0]
                self.df = pd.read_csv(io.BytesIO(uploaded[filename]), skiprows=[1], encoding="utf-8-sig")

                self.process_all_data()

                self.import_status.value = f"<p style='color:green; font-weight:bold;'>✅ 読み込み成功: {len(self.df)} 名のデータを解析しました！上の「個人診断 & 希望強化」タブを開いてください。</p>"
                display(HTML("<h4>読み込みデータプレビュー（最初の5名）:</h4>"))
                display(self.df.head())

            except Exception as e:
                self.import_status.value = f"<p style='color:red;'>❌ エラーが発生しました。ファイル形式を確認してください: {str(e)}</p>"

    def process_all_data(self):
        # UIのコントロールを有効化
        options = [(f"ID {int(row['ID'])} ({row['性別']})", int(row['ID'])) for _, row in self.df.iterrows()]
        self.player_select.options = options
        self.player_select.disabled = False
        self.target_select.disabled = False

        all_scores = []
        for _, row in self.df.iterrows():
            player_id = int(row["ID"])
            scores = calc_position_scores(row)
            score_row = {"ID": player_id, "性別": row["性別"]}
            score_row.update(scores)
            all_scores.append(score_row)

        self.score_df = pd.DataFrame(all_scores).set_index("ID")

        # チーム比較表を生成
        with self.matrix_output:
            clear_output()
            display(HTML("<p style='color:#666;'>全選手の各ポジション適性スコアです。緑色が濃いほど適性が高い（優秀）ことを示します。</p>"))
            display(
                self.score_df.style
                .background_gradient(cmap="YlGn", subset=POSITIONS, axis=None)
                .format("{:.1f}", subset=POSITIONS)
            )

        self.download_results_btn.disabled = False

        # 初期状態レポートを表示
        self.update_report()

    def on_ui_change(self, change):
        # 選手、もしくは目標ポジションが変わったらレポートを再描画
        if change['new']:
            self.update_report()

    def update_report(self):
        target_id = self.player_select.value
        target_pos = self.target_select.value

        if self.df is not None and target_id:
            # 該当する行のデータを抽出
            row = self.df[self.df["ID"] == target_id].iloc[0]
            html_content = generate_html_report(row, target_pos)

            with self.report_output:
                clear_output()
                display(HTML(html_content))

    def download_processed_csv(self, b):
        if self.df is None: return
        out_rows = []
        for _, row in self.df.iterrows():
            player_id = int(row["ID"])
            scores = calc_position_scores(row)
            r = {"ID": player_id, "性別": row["性別"]}
            r.update({f"{p}_適性スコア": scores[p] for p in POSITIONS})
            out_rows.append(r)

        out_df = pd.DataFrame(out_rows)
        out_df.to_csv("volleyball_detailed_analysis.csv", index=False, encoding="utf-8-sig")
        files.download("volleyball_detailed_analysis.csv")

# アプリの起動
app = VolleyballApp()
app.display_app()

HTML(value='\n            <div style="background: linear-gradient(135deg, #1a73e8, #0d47a1); color: white; pad…